In [17]:
import duckdb

In [18]:
con = duckdb.connect(database='dados_duckdb.db', read_only=False)

In [19]:
df = con.execute("""
                 SELECT * FROM (
                 SELECT *, ROW_NUMBER() OVER (PARTITION BY NATBR ORDER BY data_ingestao DESC) AS row FROM bronze_z0019
                 WHERE data_ingestao >= '2025-01-11') 
                 WHERE row = 1 
                 """
                 ).fetchdf()
df.head(20)

,NATBR,MAKTX,WERKS,MAINS,LABST,nome_arquivo,data_ingestao,row
0,10004,SERRA,BT20,100,200,z0019_2.csv,2026-02-22 11:52:40.401086,1
1,10003,PREGO,BT10,100,660,z0019_2.csv,2026-02-22 11:52:40.401086,1
2,10002,MARTELO,BT50,100,1500,z0019_1.csv,2026-02-22 11:32:18.811185,1
3,10001,PARAFUSO,BT10,100,100,z0019_1.csv,2026-02-22 11:32:18.811185,1
4,10005,MACHADO,BT30,100,300,z0019_2.csv,2026-02-22 11:52:40.401086,1


In [20]:
df_final = df.drop(columns=['nome_arquivo', 'data_ingestao', 'row'])
df_final = df_final.rename(columns={'NATBR': 'id', 'MAKTX': 'nm_produto', 'WERKS': 'id_categoria', 'MAINS': 'id_fornecedor', 'LABST': 'vl_preco'})
df_final.head()

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10004,SERRA,BT20,100,200
1,10003,PREGO,BT10,100,660
2,10002,MARTELO,BT50,100,1500
3,10001,PARAFUSO,BT10,100,100
4,10005,MACHADO,BT30,100,300


In [21]:
df_final.dtypes

id               object
nm_produto       object
id_categoria     object
id_fornecedor    object
vl_preco         object
dtype: object

In [27]:
df2 = df_final
df2 = df2.astype(
    {
        'id' : 'int',
        'nm_produto' : 'str',
        'id_categoria' : 'str',
        'id_fornecedor' : 'int',
        'vl_preco' : 'float'
    }
)

df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10004,SERRA,BT20,100,200.0
1,10003,PREGO,BT10,100,660.0
2,10002,MARTELO,BT50,100,1500.0
3,10001,PARAFUSO,BT10,100,100.0
4,10005,MACHADO,BT30,100,300.0


In [28]:
df2.dtypes

id                 int32
nm_produto        object
id_categoria      object
id_fornecedor      int32
vl_preco         float64
dtype: object

In [29]:
con.execute("""
CREATE TABLE IF NOT EXISTS produtos (
    id BIGINT,
    nm_produto TEXT,
    id_categoria TEXT,
    id_fornecedor BIGINT,
    vl_preco FLOAT
)
""")

In [30]:
df2.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10004,SERRA,BT20,100,200.0
1,10003,PREGO,BT10,100,660.0
2,10002,MARTELO,BT50,100,1500.0
3,10001,PARAFUSO,BT10,100,100.0
4,10005,MACHADO,BT30,100,300.0


In [33]:
con.execute("INSERT INTO produtos SELECT * FROM df2")

In [34]:
df_resultado = con.execute("select * from produtos").fetchdf()
df_resultado.head(10)

,id,nm_produto,id_categoria,id_fornecedor,vl_preco
0,10004,SERRA,BT20,100,200.0
1,10003,PREGO,BT10,100,660.0
2,10002,MARTELO,BT50,100,1500.0
3,10001,PARAFUSO,BT10,100,100.0
4,10005,MACHADO,BT30,100,300.0


In [35]:
con.close()